# Lesson 5 — Experimental Methodology

Self-assessment. No code: every answer is a sentence, a short calculation, or a
diagnosis.

Several questions ask you to *criticise* a described experiment. Those are the
ones worth spending time on — the final project is marked on exactly that skill.

Numbers quoted throughout come from the lesson's notebooks: 800 disk drives, 29
of which fail, and an energy curve with a known noise variance of 484. Scores
are the area under the receiver operating characteristic (ROC) curve
(AUC), as defined in lesson 4.

## Part 1 — A score is a measurement

**1. Lesson 1 said that holding data out and never looking at it gives an honest estimate of performance. That is true. What does it not give you?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It makes the estimate <b>unbiased</b> — centred on the truth. It says nothing about how <b>precise</b> any single measurement is.</li>
        <li>A test score is computed on a finite sample of held-out rows; a different sample would give a different number.</li>
        <li>The failure is treating "test set accuracy" as a property of the model rather than a measurement with a standard error.</li>
    </ul>
    </p>
</details>

**2. Over 200 legitimate 75/25 splits of the same 800 drives, the same model scored between 0.885 and 1.000. Explain why the best of those results is more dangerous than the worst.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A disappointing score makes you frown and keep working — it is self-correcting.</li>
        <li>A delightful score makes you <b>stop</b>: you write it down, report it, and move on.</li>
        <li>So the splits that flatter you are the ones you are least likely to interrogate. The errors that survive are the ones nobody had a reason to look for.</li>
        <li>This is a claim about attention, not about honesty.</li>
    </ul>
    </p>
</details>

**3. The test sets above each contained 7 failures. Why does that number matter more than the 200 rows in the test set, and what diagnostic question does it suggest?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The precision of a classification metric is governed by the count of the <b>rarer class</b>, not by the dataset size.</li>
        <li>An AUC computed from 7 positives has a wide sampling distribution however many negatives accompany them.</li>
        <li>The question to carry: <b>how many positive examples are in the test set?</b> Under roughly fifty, treat every metric as provisional.</li>
        <li>8,000 drives at the same 3.8% rate gives 306 positives and a stable estimate; 800 gives 29 and does not.</li>
    </ul>
    </p>
</details>

**4. Three models had mean AUC 0.9549, 0.9549 and 0.9544, and a single split declared them best 91, 71 and 38 times out of 200. What should you conclude about the models, and about the practice of selecting on one split?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>About the models: <b>they are equivalent.</b> The differences in mean are an order of magnitude smaller than the spread of a single measurement.</li>
        <li>About the practice: the win counts contain <b>no information about the models</b>. They record which model suited which random quarter of the data.</li>
        <li>Reporting a noisy number is bad; <b>choosing</b> with one is worse, because noise does not average out when it is used to decide — it decides.</li>
        <li>Run this once, as real life allows, and you report a winner with a number to support it. That conclusion is noise wearing the clothes of a result.</li>
    </ul>
    </p>
</details>

## Part 2 — Cross-validation

**5. Describe k-fold cross-validation, and identify the clause that makes it legitimate rather than "testing on the training data with extra steps".**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Cut the data into k equal parts; train on k−1 and test on the one left out; repeat until every part has served as the test set exactly once.</li>
        <li>The legitimising clause: <b>every row is predicted exactly once, by a model that never saw that row.</b></li>
        <li>No prediction is ever made by a model that was trained on the row being predicted, so no individual score is contaminated.</li>
    </ul>
    </p>
</details>

**6. Cross-validation reports one number, but k different models were fitted. What exactly does that number estimate?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The performance of the <b>procedure</b>, not of any particular fitted model — the reported number belongs to none of the k models.</li>
        <li>It answers: <i>if I apply this training procedure to a dataset of about this size, what should I expect?</i></li>
        <li>That is usually the question you actually have, since the model you ship is a k+1-th one trained on everything.</li>
    </ul>
    </p>
</details>

**7. Why is the cross-validated estimate slightly pessimistic, and what would reduce that pessimism?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Each fold trains on a fraction 1 − 1/k of the data — for k = 5, only 80% — and less training data gives a slightly worse model.</li>
        <li>So it estimates the risk of a model trained on less data than the one you will eventually deploy.</li>
        <li>The bias is in the <b>safe direction</b>, and it shrinks as k grows. Leave-one-out (k = m) minimises it.</li>
        <li>The trade is cost and, as the next question shows, a noisier estimate.</li>
    </ul>
    </p>
</details>

**8. You have five fold scores. Why should you not quote s/√5 as a confidence interval?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>That formula assumes the five scores are <b>independent</b>. They are not.</li>
        <li>For k = 5, any two training sets share three quarters of their rows, so the scores are positively correlated.</li>
        <li>The true variance is <code>σ²/k + ((k−1)/k)·ρ·σ²</code>; the naive formula keeps only the first term and so understates the uncertainty.</li>
        <li>Bengio and Grandvalet (2004) showed no unbiased estimator of this variance exists in general.</li>
        <li><b>Practical rule:</b> quote the mean and the spread, call the spread a spread, and treat small differences between models as unresolved.</li>
    </ul>
    </p>
</details>

**9. An unstratified 5-fold split of the 800 drives produced a fold containing 1 failure out of 160 rows. Why does that matter, and what is the fix?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>An AUC — or precision, or recall — computed from a single positive example is not a measurement of anything.</li>
        <li>It contributes a wildly noisy value to the average, and nothing in the code warns you.</li>
        <li>The fix is <code>StratifiedKFold</code>, which keeps each fold's class balance equal to the whole dataset's. It costs nothing.</li>
        <li><b>Stratify whenever a class is rare</b> — which, from lesson 4, is whenever the problem is interesting.</li>
    </ul>
    </p>
</details>

**10. Your table holds ten telemetry readings per drive, and you use a random 5-fold split. The cross-validated score is excellent. Explain why you should not believe it, and what to use instead.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A random split puts nine of a drive's readings in training and one in test, so the test row is nearly a duplicate of a training row.</li>
        <li>The model can score well by <b>recognising the drive</b> rather than by learning anything about failure.</li>
        <li>It will be worthless on a drive it has never seen — which is the only situation that matters.</li>
        <li>Use <code>GroupKFold</code> with the drive identifier as the group, so no drive appears in both training and test.</li>
        <li>The general rule: <b>split along the axis you must generalise across.</b></li>
    </ul>
    </p>
</details>

**11. Why is a random split wrong for time-ordered data, even when rows are otherwise independent?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A random split trains on Wednesday and tests on Tuesday. No deployment ever works that way.</li>
        <li>Real use always predicts <b>forward</b>, so validation must too, or the estimate answers a question nobody asked.</li>
        <li><code>TimeSeriesSplit</code> trains on a prefix and tests on what follows.</li>
        <li>The symptom of getting this wrong is the classic one: excellent validation, poor production.</li>
    </ul>
    </p>
</details>

## Part 3 — Bias, variance and the noise floor

**12. Distinguish bias from variance in terms of what happens when you fit the same model to many different training sets.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Bias:</b> the fitted models agree with each other and the average is far from the truth — consistently, confidently wrong. A straight line fitted to a curve.</li>
        <li><b>Variance:</b> the fitted models disagree wildly with each other — unreliable, whatever the average looks like. A degree-12 polynomial on 25 points.</li>
        <li><b>Noise:</b> uncertainty in the target itself. Not a property of the model at all, and no model removes it.</li>
    </ul>
    </p>
</details>

**13. State the decomposition and say why the cross terms vanish in its derivation.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>E[(y − f̂)²] = bias² + variance + σ²</code>, at a fixed test point, in expectation over training sets and over the noise in the new observation.</li>
        <li>The first cross term vanishes because ε is <b>independent of the training set</b> and has mean zero.</li>
        <li>The second vanishes because <code>E[f̂] = f̄</code> by definition of the average model, so <code>E[f̄ − f̂] = 0</code>.</li>
        <li>It is an <b>identity</b>, not an approximation — notebook 2 confirms it on 300 training sets to within 3×10⁻¹².</li>
    </ul>
    </p>
</details>

**14. At degree 12 on 25 observations, both bias² (153,537) and variance (32,174,345) were enormous. Doesn't a flexible model have low bias by definition? Resolve the apparent contradiction.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The bias term is the distance from the truth to the <b>average</b> fitted model.</li>
        <li>When variance is this large, the fitted curves fly off in different directions — especially where the data runs out — so their average is not a meaningful curve.</li>
        <li>The bias term therefore inherits the instability rather than measuring rigidity.</li>
        <li><b>Read the two numbers together.</b> Variance dominates by four orders of magnitude, and that is the diagnosis.</li>
    </ul>
    </p>
</details>

**15. The noise variance was 484 at every complexity. State the practical diagnostic that follows.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The noise floor is a property of the data, not the model; no model, algorithm or quantity of data brings expected error below it.</li>
        <li><b>A reported error below the noise floor is evidence of contamination, not of excellence.</b></li>
        <li>If you know your measurement precision and your model beats it, something has leaked. This is a real check in engineering and the physical sciences, and it catches errors nothing else catches.</li>
    </ul>
    </p>
</details>

**16. The best polynomial degree was 2 at n = 25 and 5 at n = 60. What general lesson follows, and what common frustration does it explain?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>"Which model is best?" is not a question about models.</b> It is a question about models <i>and</i> the quantity of data available.</li>
        <li>More data lowers the variance of a flexible model, so the optimum moves towards more complexity.</li>
        <li>It explains the frustration of a published result that does not reproduce: a large model beat a small one on a far bigger dataset. Neither experiment was wrong; the conclusion simply does not transfer.</li>
    </ul>
    </p>
</details>

## Part 4 — Learning curves

**17. A learning curve shows training and validation scores meeting at 0.715 and 0.718, flat from the first point to the last. Diagnose it, and say what you would and would not do.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>High bias.</b> The curves meet, and they meet low; the model is equally mediocre on data it has seen and data it has not.</li>
        <li><i>Would:</i> a more flexible model, better features, less regularisation.</li>
        <li><i>Would not:</i> collect more data. The curve is already flat — more rows land on the same plateau.</li>
        <li>This is the expensive mistake, because collecting data is the slowest and costliest intervention, and the plot says in advance it will buy nothing.</li>
    </ul>
    </p>
</details>

**18. Another curve shows training 1.000 against validation 0.847, with the validation curve still rising at the right edge. Diagnose it and prescribe.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>High variance.</b> A wide gap, and a training score of exactly 1.000 means the model separates its training set perfectly.</li>
        <li>In the lesson this came from 6 real features plus <b>150 columns of pure noise</b>.</li>
        <li><i>Prescribe:</i> more data — the rising curve is the evidence that it will help — or stronger regularisation, fewer features, a simpler model.</li>
    </ul>
    </p>
</details>

**19. State the one-line diagnostic for reading a learning curve, and name the predictable mistake it prevents.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Is the gap large, and is the validation curve still rising?</b> If yes, get more data. If the curves have met and levelled off, more data is money spent on nothing.</li>
        <li>The predictable mistake is <b>prescribing more data for a bias problem</b>.</li>
        <li>The instinct behind it is entirely sound: more data almost always helps and never makes a model worse. It simply does not help <i>that</i> failure — and fifteen lines of code tell you which failure you have.</li>
    </ul>
    </p>
</details>

## Part 5 — Leakage that survives cross-validation

**20. Selecting the 10 best of 2,000 pure-noise columns and then cross-validating gave AUC 0.931 with tight fold agreement. Identify the error precisely, and say what cross-validation did wrong.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The error: <code>SelectKBest</code> was fitted on <b>all the rows</b>, including those that later served as test folds.</li>
        <li>It searched 2,000 columns for the ones matching labels it had already seen, then handed the winners to cross-validation.</li>
        <li><b>Cross-validation did nothing wrong.</b> It faithfully measured a procedure that had already peeked — it was lied to, it did not fail.</li>
        <li>Tight fold agreement makes it worse, not better: the result looks trustworthy.</li>
    </ul>
    </p>
</details>

**21. Moving the selection inside the Pipeline brought the estimate down to 0.658 — better, but not 0.500. Explain why not, and say what would actually fix it.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>With 2,000 columns and 800 rows, some columns correlate with the label <b>by accident across the whole dataset</b>.</li>
        <li>That accident is a property of <b>this sample</b>, so it is present in every subset — every training fold and every test fold alike.</li>
        <li>A selector fitted honestly on four fifths finds those columns, and they still "work" on the remaining fifth, because the spurious correlation was never fold-specific.</li>
        <li><b>Cross-validation protects against leaking between folds, not against searching a large space on a small sample.</b></li>
        <li>What fixes it: a test set held out before any of this and used once; fewer candidates; or more rows. Nothing else.</li>
    </ul>
    </p>
</details>

**22. A grid search over 25 combinations reported best_score_ = 0.7999 on signal-free data, where the average candidate scored 0.7265. Why is best_score_ biased, and by how much was it wrong here?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><code>best_score_</code> is the <b>maximum of 25 noisy estimates</b>, and the maximum of a set of noisy numbers is biased upward even when every one measures the same quantity.</li>
        <li>It reports the performance of <b>whichever configuration got luckiest</b>, not of the configuration chosen.</li>
        <li>Nested cross-validation gave 0.6699, so the optimism was <b>+0.13</b> — larger than most differences anyone reports between competing methods.</li>
        <li>And even 0.67 is above the truth of 0.500, for the reason in question 21.</li>
    </ul>
    </p>
</details>

**23. Describe nested cross-validation, and state one thing it is not for.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Outer loop</b> holds out a fifth. <b>Inner loop</b> runs the entire search on the remaining four fifths. The search's chosen model is scored on the held-out fifth. Repeat five times.</li>
        <li>The inner loop may overfit its own data as much as it likes; the outer block was never part of it.</li>
        <li><b>It is not a way to choose hyperparameters</b> — it produces k possibly different winners.</li>
        <li>It estimates <i>what choosing costs</i>. You choose on all the data afterwards, and report the nested figure.</li>
    </ul>
    </p>
</details>

**24. In lesson 4 the threshold was chosen on the test set and cost 3,300 EUR; chosen honestly on a validation set it cost 4,700 EUR on the same test set. Which number is reportable, and why is the worse one the right one?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>4,700 EUR</b> is reportable.</li>
        <li>The 3,300 figure describes a threshold tuned to <i>this particular test set</i>; there is no reason it survives contact with the next 200 drives.</li>
        <li>The 4,700 figure is an <b>unbiased estimate of what happens next</b>, which is the only thing a report is for.</li>
        <li>Note the price of doing it properly: training now uses 56% of the data rather than 75%, because the validation set has to come from somewhere. That is the argument for choosing by cross-validation on the training portion and keeping the test set whole.</li>
    </ul>
    </p>
</details>

## Part 6 — Reproducibility, and reading critically

**25. Thirty seeds, each perfectly reproducible, gave scores from 0.913 to 1.000. What does a fixed seed buy, and what does it not?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>It buys <b>repeatability</b>: anyone with the same code gets the same number.</li>
        <li>It does not buy <b>stability</b>: those thirty reproducible numbers disagree by nearly nine points.</li>
        <li>So reproducibility is necessary and nowhere near sufficient. <b>Report the seed and the spread.</b></li>
        <li>"AUC 1.000 (random_state=3)" is fully reproducible and thoroughly misleading; "AUC 0.951 ± 0.019 over 5-fold CV, seed 0" is reproducible and honest.</li>
    </ul>
    </p>
</details>

**26. Besides seeds, what else must be recorded for a result to be reproducible in two years?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Library versions.</b> numpy, pandas and scikit-learn change defaults between releases, which has invalidated published comparisons.</li>
        <li><b>What you could not fix</b> — thread scheduling, GPU non-determinism — named explicitly in the report.</li>
        <li><b>The code that actually produced the number</b>, not a tidied-up version of it.</li>
        <li>The course container exists for the first of these: it pins the environment so your result and your marker's are the same result.</li>
    </ul>
    </p>
</details>

**27. List the four things that belong in any report of a model's performance, and say which failure each one guards against.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>The metric, the spread, and how it was estimated</b> — guards against the split lottery.</li>
        <li><b>How many positives the test set held</b> — explains why the spread is what it is.</li>
        <li><b>Every seed and every library version</b> — guards against irreproducibility.</li>
        <li><b>Which choices were made on which data</b> — guards against the optimism of selection.</li>
        <li>Read papers with the same list: a lone figure with no spread, no fold count and no statement of how hyperparameters were chosen leaves all four questions open.</li>
    </ul>
    </p>
</details>

**28. A colleague reports: "We tried 40 model configurations, cross-validated each, and the best gave AUC 0.94 on our 600-row dataset." Write the questions you would ask.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>Is 0.94 the best of 40 cross-validated scores?</b> If so it is the maximum of 40 noisy estimates and biased upward — ask for a nested estimate.</li>
        <li><b>How many positives are in the data?</b> On 600 rows the answer governs the precision of everything.</li>
        <li><b>What is the spread across folds</b>, and how many folds?</li>
        <li><b>Was any preprocessing, imputation or feature selection fitted before the split?</b></li>
        <li><b>Are the rows independent?</b> Repeated measurements of the same entity, or time order, would require GroupKFold or TimeSeriesSplit.</li>
        <li><b>Is there a held-out test set that has been used once</b>, or is 0.94 the only number that exists?</li>
    </ul>
    </p>
</details>

**29. Summarise the shape shared by nearly every error in this lesson, in one sentence.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>A choice was made by looking at data that was then used to report the result.</b></li>
        <li>Choosing a threshold on the test set; selecting features on the whole dataset; reporting the best of many searched configurations; picking the seed whose score you liked.</li>
        <li>The cure is always the same shape too: make the choice on data reserved for choosing, and report on data used once.</li>
    </ul>
    </p>
</details>